# preloading

In [1]:
import os
from pathlib import Path
import datetime
import time
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
import functools
import itertools

#os.environ["PYLAB_DB_LOCAL"] = ""
#os.environ["PYLAB_DB_OUT"] = ""
import pyflexlab
from pyomnix.utils import CM_TO_INCH, next_lst_gen, constant_generator
from pyomnix import DataManipulator
from pyflexlab import FileOrganizer, DataProcess, MeasureFlow

from pyflexlab.measure_flow import MeasurementRecipe, PlotRecipe
from pyomnix.omnix_logger import get_logger
# Measure module will need the NIDAQmx module to work,

2026-06-09 00:23:38 - pyomnix.consts - INFO - ^_^ read from OMNIX_PATH: C:\Users\Dongkai\OneDrive\omnix-path
2026-06-09 00:23:38 - pyflexlab.constants - INFO - set with PYLAB_DB_LOCAL_DONGKAI
2026-06-09 00:23:38 - pyflexlab.constants - INFO - set with PYLAB_DB_OUT_DONGKAI
2026-06-09 00:23:38 - pyflexlab.constants - INFO - read from PYLAB_DB_LOCAL: C:\Users\Dongkai\OneDrive - Nanyang Technological University\script-tools\data_files
2026-06-09 00:23:38 - pyflexlab.constants - INFO - read from PYLAB_DB_OUT: C:\Users\Dongkai\OneDrive - Nanyang Technological University\database


In [2]:
project_name = "Date-Material"
folder = DataProcess(project_name)
measureflow = MeasureFlow(project_name)
plotobj = DataManipulator(2)
#Folder.add_measurement("RT")
#Folder.add_plan("RT","__whole, measure the whole RT")

In [3]:
folder.tree()

Date-Material

0 directories


### Add measurement type if needed

In [ ]:
#FileOrganizer.add_measurement_type("V_source_sweep_ac","Vmax{maxv}V-step{stepv}V-freq{freq}Hz-{vhigh}-{vlow}", overwrite=False)

In [ ]:
#FileOrganizer.name_fstr_gen("I_source_sweep_dc","V_sense","T_vary")

('I-V-T',
 'Imax{maxi}A-step{stepi}A-{iin}-{iout}-swpmode{mode}_V{note}-{vhigh}-{vlow}_Temp{Tstart}-{Tstop}K')

In [ ]:
#DataPlot.gui_pan_color()

# Setup and Parameter

In [3]:
#measureflow.get_visa_resources()   # list all VISA resources
#measureflow.load_meter("6221","GPIB0::12::INSTR")
#measureflow.load_meter("2182","GPIB0::7::INSTR")
#measureflow.load_meter("2400","GPIB0::23::INSTR")
#measureflow.load_meter("6430","GPIB0::24::INSTR")
#measureflow.load_meter("sr830","GPIB0::8::INSTR","GPIB0::9::INSTR")
#measureflow.load_mercury_ips("TCPIP0::10.97.24.237::7020::SOCKET")
#measureflow.load_mercury_itc("TCPIP0::10.94.28.24::7020::SOCKET")
#measureflow.load_ITC503("GPIB0::23::INSTR","GPIB0::24::INSTR")
#measureflow.load_rotator()
measureflow.load_fakes(3)

*****
# Guidance for measurement

### *sweep modes:*
- DC
    - 0-max-0
    - 0--max-max-0
    - manual
- AC
    - 0-max-0
    - 0-max
    - manual

### Column Names:
- time (default in generator)
- X_source
- X (sense->ext) 
    - (if lock-in, then X, Y, R, Theta)

### Combination of Varies or Mappings
- set `if_combine_gen=False` in `get_mea_dict` to get the list of generators instead of a whole list generator
- use `constants.next_lst_gen` to get the next values list
- replace mapping generators with constant generators (careful about the values) to temporarily pause the mapping and do varying, and remember to vary circularly (hysteretically) use `reverse=True`

******

# Example Usage
For more usage and customization, please refer to the `measure_flow.py` file and documentation.

In [ ]:
measureflow.measure_Vswp_I_vicurve_legacy(
    vmax=0.5,
    vstep=0.005,
    high=7,
    low=4,
    swpmode="0--max-max-0",
    meter=measureflow.instrs["2400"][0],
    compliance="1uA",
    folder_name="contact-test",
    step_time=0.4,
    plotobj=plotobj,
)

In [ ]:
measureflow.measure_VswpVswp_II_BT_dsgatemapping_legacy(
    constrained = False,
    vds_max=0.01,
    ds_map_lst = np.arange(0, 0.1, 0.001),
    ds_high=0,
    ds_low=0,
    ds_meter=measureflow.instrs["2400"][0],
    ds_compliance="1uA",
    vg=0,
    gate_map_lst = np.concatanete([np.arange(0, 50, 0.2), np.arange(50, 0, -0.2)]),
    vg_high=0,
    vg_meter=measureflow.instrs["6430"][0],
    vg_compliance="5nA",
    field=0,
    temperature=2,
    folder_name="",
    step_time = 1,
    individual_plot = True,
    ds_gate_order = (0, 1),
    calculate_from_ds = None,
    plotobj = plotobj,
)

In [ ]:
ids = "1uA"
ds_high = 0
ds_low = 0
sense_high = None
sense_low = None
ds_meter = measureflow.instrs["2400"][0]
ds_compliance = "1V"
freq = None
vg = 0
vg_high = 0
vg_meter = measureflow.instrs["6430"][0]
vg_compliance = "5nA"
field_start = -1
field_end = 1
temperature = 2
folder_name = "rh-loop"
step_time = 0.3
wait_before_vary = 5
vary_loop = True
if_plot = True
saving_interval = 7
rh_loop_plotobj = plotobj
use_dash = False


{'ips': <pyflexlab.equip_wrapper.simulated.FakeMag at 0x27f494657f0>,
 'itc': <pyflexlab.equip_wrapper.simulated.SimITC at 0x27f49465a90>,
 'fakes': [<pyflexlab.equip_wrapper.simulated.SimMeter at 0x27f49465be0>,
  <pyflexlab.equip_wrapper.simulated.SimMeter at 0x27f493e7750>]}

{'begin_vary': True, 'vary_lst': []}

In [ ]:
# Recipe-flow version of measure_IV_VI_BvaryT_rhloop_legacy.
# This intentionally expands the full legacy flow here instead of using a wrapper/builder.
#ds_src_sens_lst[1].four_wire = True

rh_loop_state = {"begin_vary": False, "vary_lst": []}

def rh_loop_after_prepare(mea_dict):
    print("filepath: %s", mea_dict["file_path"])
    print("no of columns(with time column): %d", mea_dict["record_num"])
    print("vary modules: %s", mea_dict["vary_mod"])
    #rh_loop_state["vary_lst"], _, _, _ = measureflow._extract_vary(mea_dict)

def vary_on_record(_):
    if not rh_loop_state["begin_vary"]:
        for funci in rh_loop_state["vary_lst"]:
            funci()
        rh_loop_state["begin_vary"] = True

def rh_loop_plot_update(plotobj, gen_i):
    plotobj.live_plot_update(
        [0, 0, 0],
        [0, 1, 2],
        [0, 0, 0],
        [gen_i[5], gen_i[5], gen_i[0]],
        [gen_i[3], gen_i[6], gen_i[5]],
        incremental=True,
    )
    plotobj.live_plot_update(0, 2, 0, gen_i[0], gen_i[5], incremental=True)

rh_loop_recipe = MeasurementRecipe(
    measure_mods=(
        "I_source_fixed_dc",
        "V_source_fixed_dc",
        "V_sense_dc",
        "I_sense_dc",
        "B_vary",
        "T_fixed",
    ),
    args=(
        1,
        0,
        0,
        2,
        0,
        0,
        "",
        0,
        0,
        "",
        0,
        0,
        -1,
        1,
        0,
    ),
    wrapper_lst=[
        measureflow.instrs["fakes"][0],
        measureflow.instrs["fakes"][1],
        measureflow.instrs["fakes"][0],
        measureflow.instrs["fakes"][1],
    ],
    compliance_lst=[1, 1],
    measure_kwargs={
        "if_combine_gen": True,
        "special_name": "specialname",
        "measure_nickname": "nickmea",
        "vary_loop": True,
        "source_wait": 0.2
    },
    step_time=0.1,
    plot=PlotRecipe(
        plotobj= DataManipulator(3),
        init_args=(1, 3, 1),
        init_kwargs={"titles": [["B I", "B T", "t B"]]},
        update=rh_loop_plot_update,
        saving_interval=3,
        inline_jupyter=False,
    ),
    after_prepare=rh_loop_after_prepare,
    on_record=vary_on_record
)


rh_loop_result = measureflow.run_recipe(rh_loop_recipe)

In [7]:
mea_dict = measureflow.prepare_recipe(rh_loop_recipe)

2026-06-09 00:31:56 - pyflexlab.file_organizer - WARNING - IV-VI-BT is already in the project record file.
2026-06-09 00:31:56 - pyflexlab.measure_manager - INFO - Filename is: Ifix1A-0-0-Vfix2V-0-0_V-0-0-I-0-0_B-1-1T-Temp0K.csv
2026-06-09 00:31:57 - pyflexlab.measure_manager - WARNING - C:\Users\Dongkai\OneDrive - Nanyang Technological University\database\Date-Material\nickmea\IV-VI-BT\specialname\Ifix1A-0-0-Vfix2V-0-0_V-0-0-I-0-0_B-1-1T-Temp0K.csv already exists, will be overwritten
2026-06-09 00:31:57 - pyflexlab.measure_manager - WARNING - C:\Users\Dongkai\OneDrive - Nanyang Technological University\database\Date-Material\nickmea\IV-VI-BT\specialname\Ifix1A-0-0-Vfix2V-0-0_V-0-0-I-0-0_B-1-1T-Temp0K.csv has been backed up to C:\Users\Dongkai\OneDrive - Nanyang Technological University\database\Date-Material\nickmea\IV-VI-BT\specialname\Ifix1A-0-0-Vfix2V-0-0_V-0-0-I-0-0_B-1-1T-Temp0K.csv.bak


In [8]:
mea_dict

{'gen_lst': <generator object combined_generator_list at 0x0000026780C5D210>,
 'swp_idx': [],
 'file_path': ShPath('C:/Users/Dongkai/OneDrive - Nanyang Technological University/database/Date-Material/nickmea/IV-VI-BT/specialname/Ifix1A-0-0-Vfix2V-0-0_V-0-0-I-0-0_B-1-1T-Temp0K.csv'),
 'plot_record_path': ShPath('C:/Users/Dongkai/OneDrive - Nanyang Technological University/database/Date-Material/plot/nickmea/IV-VI-BT/specialname/Ifix1A-0-0-Vfix2V-0-0_V-0-0-I-0-0_B-1-1T-Temp0K.png'),
 'record_num': 7,
 'vary_mod': ['B'],
 'tmp_vary': None,
 'mag_vary': (<function pyflexlab.measure_manager.MeasureManager.get_measure_dict.<locals>.mag_vary(reverse: bool = False, oth_mod={'name': 'B', 'sweep_fix': 'vary', 'fix': None, 'start': -1, 'stop': 1, 'step': None, 'mode': None})>,
  <function pyflexlab.measure_manager.MeasureManager.get_measure_dict.<locals>.<lambda>()>,
  <function pyflexlab.measure_manager.MeasureManager.get_measure_dict.<locals>.<lambda>()>,
  (-1, 1)),
 'angle_vary': None}